In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load original cleaned data (has all features)
df_original = pd.read_csv('telco_customer_churn_cleaned.csv')

# Load churn probabilities from Phase 4
df_churn = pd.read_csv('telco_churn_probability.csv')

# Merge: churn probability + original features
clv_dataset = df_original.merge(
    df_churn[['customerID', 'churn_probability']], 
    on='customerID', 
    how='inner'
)

print(f"\n✓ Loaded dataset with {clv_dataset.shape[0]:,} customers")
print(f"✓ Columns: {clv_dataset.shape[1]}")
print(f"\nColumn names:")
for col in clv_dataset.columns:
    print(f"  - {col}")


✓ Loaded dataset with 5,636 customers
✓ Columns: 32

Column names:
  - customerID
  - SeniorCitizen
  - tenure
  - MonthlyCharges
  - TotalCharges
  - churn_flag
  - partner_flag
  - dependents_flag
  - phoneservice_flag
  - paperlessbilling_flag
  - contract_ordinal
  - gender_Male
  - internet_Fiber optic
  - internet_No
  - multiplelines_No phone service
  - multiplelines_Yes
  - onlinesecurity_No internet service
  - onlinesecurity_Yes
  - onlinebackup_No internet service
  - onlinebackup_Yes
  - deviceprotection_No internet service
  - deviceprotection_Yes
  - techsupport_No internet service
  - techsupport_Yes
  - streamingtv_No internet service
  - streamingtv_Yes
  - streamingmovies_No internet service
  - streamingmovies_Yes
  - payment_Credit card (automatic)
  - payment_Electronic check
  - payment_Mailed check
  - churn_probability


In [2]:
print("Columns in df_original:")
print(df_original.columns.tolist())
print("\n\nColumns in clv_dataset after merge:")
print(clv_dataset.columns.tolist())

Columns in df_original:
['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'churn_flag', 'partner_flag', 'dependents_flag', 'phoneservice_flag', 'paperlessbilling_flag', 'contract_ordinal', 'gender_Male', 'internet_Fiber optic', 'internet_No', 'multiplelines_No phone service', 'multiplelines_Yes', 'onlinesecurity_No internet service', 'onlinesecurity_Yes', 'onlinebackup_No internet service', 'onlinebackup_Yes', 'deviceprotection_No internet service', 'deviceprotection_Yes', 'techsupport_No internet service', 'techsupport_Yes', 'streamingtv_No internet service', 'streamingtv_Yes', 'streamingmovies_No internet service', 'streamingmovies_Yes', 'payment_Credit card (automatic)', 'payment_Electronic check', 'payment_Mailed check']


Columns in clv_dataset after merge:
['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'churn_flag', 'partner_flag', 'dependents_flag', 'phoneservice_flag', 'paperlessbilling_flag', 'contract_ordinal', 'gender_Ma

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 2: CREATE NUM_SERVICES (count of adopted services)
# ══════════════════════════════════════════════════════════════════════════
# Service columns: phoneservice, onlinesecurity, onlinebackup, deviceprotection,
#                  techsupport, streamingtv, streamingmovies

service_cols = ['phoneservice_flag', 'onlinesecurity_Yes', 'onlinebackup_Yes', 
                'deviceprotection_Yes', 'techsupport_Yes', 'streamingtv_Yes', 'streamingmovies_Yes']

clv_dataset['num_services'] = clv_dataset[service_cols].sum(axis=1)

print(f"\nCreated num_services (count of adopted services)")
print(f"  Range: {clv_dataset['num_services'].min():.0f} - {clv_dataset['num_services'].max():.0f}")
print(f"  Mean: {clv_dataset['num_services'].mean():.2f}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2B: IDENTIFY CLV COMPONENTS FROM AVAILABLE DATA
# ══════════════════════════════════════════════════════════════════════════

print(f"\n1. REVENUE SIGNALS:")
revenue_cols = ['MonthlyCharges', 'TotalCharges', 'tenure']
for col in revenue_cols:
    if col in clv_dataset.columns:
        print(f"  {col:20s} - Available (Mean: ${clv_dataset[col].mean():.2f})")
    else:
        print(f"  {col:20s} - Not found")

# Engagement/Stickiness Components

print(f"\n2. ENGAGEMENT SIGNALS (Stickiness):")

engagement_cols = ['num_services', 'contract_ordinal']

for col in engagement_cols:
    if col in clv_dataset.columns:
        print(f"  {col:20s} - Available (Mean: {clv_dataset[col].mean():.3f})")
    else:
        print(f"  {col:20s} - Not found")

# Risk Components

print(f"\n3. RETENTION/RISK SIGNALS:")

risk_cols = ['churn_probability']

for col in risk_cols:
    if col in clv_dataset.columns:
        print(f"  {col:20s} - Available (Mean: {clv_dataset[col].mean():.3f})")
    else:
        print(f"  {col:20s} - Not found")

print(f"\n✓ All key components available for CLV calculation")


Created num_services (count of adopted services)
  Range: 0 - 7
  Mean: 2.59

1. REVENUE SIGNALS:
  MonthlyCharges       - Available (Mean: $61.97)
  TotalCharges         - Available (Mean: $1555.61)
  tenure               - Available (Mean: $23.45)

2. ENGAGEMENT SIGNALS (Stickiness):
  num_services         - Available (Mean: 2.588)
  contract_ordinal     - Available (Mean: 0.457)

3. RETENTION/RISK SIGNALS:
  churn_probability    - Available (Mean: 0.312)

✓ All key components available for CLV calculation


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 3: CREATE DERIVED FEATURES FOR CLV MODELING
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("CREATING DERIVED CLV FEATURES")
print("="*60)

# Copy dataset for feature engineering
clv_data = clv_dataset.copy()

# ──────────────────────────────────────────────────────────────────────────
# Feature 1: Lifetime Spend (past revenue = what they've already paid)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Shows customer investment to date (higher = more valuable relationship)
clv_data['lifetime_spend'] = clv_data['TotalCharges']

print(f"\n1. LIFETIME_SPEND (Total Charges to Date)")
print(f"   What: Historical cumulative revenue from this customer")
print(f"   Why: Reflects total relationship investment")
print(f"   Range: ${clv_data['lifetime_spend'].min():.2f} - ${clv_data['lifetime_spend'].max():.2f}")
print(f"   Mean: ${clv_data['lifetime_spend'].mean():.2f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 2: Average Monthly Spend (current revenue rate)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Shows current revenue stream (annualize to get future potential)
clv_data['avg_monthly_spend'] = clv_data['MonthlyCharges']

print(f"\n2. AVG_MONTHLY_SPEND (Monthly Charges)")
print(f"   What: Current monthly revenue from this customer")
print(f"   Why: Basis for projecting future revenue")
print(f"   Range: ${clv_data['avg_monthly_spend'].min():.2f} - ${clv_data['avg_monthly_spend'].max():.2f}")
print(f"   Mean: ${clv_data['avg_monthly_spend'].mean():.2f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 3: Retention Rate (from churn probability)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Use churn probability to weight future revenue expectations
clv_data['retention_rate'] = 1 - clv_data['churn_probability']

print(f"\n3. RETENTION_RATE (1 - Churn Probability)")
print(f"   What: Probability customer stays (complement of churn risk)")
print(f"   Why: Use to weight future revenue expectations")
print(f"   Range: {clv_data['retention_rate'].min():.3f} - {clv_data['retention_rate'].max():.3f}")
print(f"   Mean: {clv_data['retention_rate'].mean():.3f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 4: Revenue Stability (monthly vs total ratio)
# ──────────────────────────────────────────────────────────────────────────
# WHY: High ratio = recent high spending; Low ratio = already paid off early years
clv_data['revenue_stability'] = clv_data['MonthlyCharges'] / (clv_data['TotalCharges'] + 1)

print(f"\n4. REVENUE_STABILITY (Monthly / Total Charges)")
print(f"   What: Ratio of current to historical spending")
print(f"   Why: Indicates if customer spending is increasing or declining")
print(f"   Range: {clv_data['revenue_stability'].min():.4f} - {clv_data['revenue_stability'].max():.4f}")
print(f"   Mean: {clv_data['revenue_stability'].mean():.4f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 5: Engagement Level (tenure + num_services normalized)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Customers with more services are more sticky (harder to leave)
tenure_norm = clv_data['tenure'] / clv_data['tenure'].max()
services_norm = clv_data['num_services'] / clv_data['num_services'].max()
clv_data['engagement_level'] = (tenure_norm + services_norm) / 2

print(f"\n5. ENGAGEMENT_LEVEL (Tenure + Service Count)")
print(f"   What: Combined tenure and service adoption score (0-1)")
print(f"   Why: Higher engagement = stickier customers = lower churn")
print(f"   Range: {clv_data['engagement_level'].min():.3f} - {clv_data['engagement_level'].max():.3f}")
print(f"   Mean: {clv_data['engagement_level'].mean():.3f}")

print(f"\n✓ All CLV features created successfully")


CREATING DERIVED CLV FEATURES

1. LIFETIME_SPEND (Total Charges to Date)
   What: Historical cumulative revenue from this customer
   Why: Reflects total relationship investment
   Range: $18.80 - $7049.50
   Mean: $1555.61

2. AVG_MONTHLY_SPEND (Monthly Charges)
   What: Current monthly revenue from this customer
   Why: Basis for projecting future revenue
   Range: $18.25 - $117.45
   Mean: $61.97

3. RETENTION_RATE (1 - Churn Probability)
   What: Probability customer stays (complement of churn risk)
   Why: Use to weight future revenue expectations
   Range: 0.078 - 0.994
   Mean: 0.688

4. REVENUE_STABILITY (Monthly / Total Charges)
   What: Ratio of current to historical spending
   Why: Indicates if customer spending is increasing or declining
   Range: 0.0155 - 0.9903
   Mean: 0.1915

5. ENGAGEMENT_LEVEL (Tenure + Service Count)
   What: Combined tenure and service adoption score (0-1)
   Why: Higher engagement = stickier customers = lower churn
   Range: 0.008 - 1.000
   Mean

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 4: EXPLAIN THE CLV CALCULATION LOGIC
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("WHY THESE FEATURES MATTER FOR CLV")
print("="*60)

explanation = """
┌─────────────────────────────────────────────────────────────┐
│ KEY INSIGHT: Past Revenue ≠ Future Value                   │
└─────────────────────────────────────────────────────────────┘

TRADITIONAL MISTAKE:
  "Customer paid $3,000 total → CLV = $3,000"
  WRONG! This ignores future churn risk

CORRECT APPROACH (What we're doing):
  1. How much has customer spent? ($3,000) → HISTORICAL value
  2. How much do they spend per month? ($100) → REVENUE RATE
  3. How likely are they to stay? (80%) → RETENTION probability
  4. Therefore: Expected future value = $100/month × 12 × 0.8 = ~$960

┌─────────────────────────────────────────────────────────────┐
│ WHY ENGAGEMENT MATTERS FOR CLV                              │
└─────────────────────────────────────────────────────────────┘

CUSTOMER A: Trial Period Customer
  - tenure: 2 months
  - num_services: 1 (phone only)
  - monthly_charges: $20
  - churn_probability: 0.50 (HIGH RISK!)
  
  Traditional CLV: $20 × 2 = $40
  Smart CLV: $20/month × 12 × 0.5 = $120 (but needs retention!)
  Action: INVEST in retention (cross-sell, loyalty program)

CUSTOMER B: Established, Multi-Service
  - tenure: 36 months
  - num_services: 4 (phone + internet + TV + security)
  - monthly_charges: $150
  - churn_probability: 0.05 (LOW RISK!)
  
  Traditional CLV: $150 × 36 = $5,400
  Smart CLV: $150/month × 12 × 0.95 = $1,710/year + expansion potential
  Action: PROTECT (maintain quality, upsell premium services)

┌─────────────────────────────────────────────────────────────┐
│ FEATURES THAT PREDICT HIGH CLV                              │
└─────────────────────────────────────────────────────────────┘

✓ HIGH engagement_level (long tenure + many services)
  → Hard to switch, integrated into daily life
  
✓ HIGH monthly_charges (revenue per user)
  → Already invested in premium services
  
✓ LOW churn_probability (high retention_rate)
  → Stable, proven loyalty
  
✓ HIGH tenure_normalized (years of relationship)
  → Habit formation, switching costs

✗ LOW features indicate churn risk
  → Trial period customers need intervention
  → New, single-service customers vulnerable
"""

print(explanation)

print(f"\n✓ CLV framework defined and ready for Phase 5 Part 2")
print(f"  Next: Build actual CLV score combining these features")


WHY THESE FEATURES MATTER FOR CLV

┌─────────────────────────────────────────────────────────────┐
│ KEY INSIGHT: Past Revenue ≠ Future Value                   │
└─────────────────────────────────────────────────────────────┘

TRADITIONAL MISTAKE:
  "Customer paid $3,000 total → CLV = $3,000"
  WRONG! This ignores future churn risk

CORRECT APPROACH (What we're doing):
  1. How much has customer spent? ($3,000) → HISTORICAL value
  2. How much do they spend per month? ($100) → REVENUE RATE
  3. How likely are they to stay? (80%) → RETENTION probability
  4. Therefore: Expected future value = $100/month × 12 × 0.8 = ~$960

┌─────────────────────────────────────────────────────────────┐
│ WHY ENGAGEMENT MATTERS FOR CLV                              │
└─────────────────────────────────────────────────────────────┘

CUSTOMER A: Trial Period Customer
  - tenure: 2 months
  - num_services: 1 (phone only)
  - monthly_charges: $20
  - churn_probability: 0.50 (HIGH RISK!)

  Traditional CLV: $2

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 5: SUMMARY STATISTICS OF CLV FEATURES
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("CLV DATASET READY FOR PHASE 5 PART 2")
print("="*60)

print(f"\nDataset shape: {clv_data.shape[0]:,} customers × {clv_data.shape[1]} features")

print(f"\nNew CLV Features Created:")
clv_features = ['lifetime_spend', 'avg_monthly_spend', 'retention_rate', 
                'revenue_stability', 'engagement_level']
for feat in clv_features:
    if feat in clv_data.columns:
        print(f"  ✓ {feat}")

print(f"\nFeature Statistics:")
feature_stats = clv_data[[col for col in clv_features if col in clv_data.columns]].describe()
print(feature_stats.to_string())

print(f"\n Phase 5 Part 1 Complete")
print(f"  Dataset ready for CLV scoring in Phase 5 Part 2")
print(f"  Variables available: clv_data (with all features)")


CLV DATASET READY FOR PHASE 5 PART 2

Dataset shape: 5,636 customers × 38 features

New CLV Features Created:
  ✓ lifetime_spend
  ✓ avg_monthly_spend
  ✓ retention_rate
  ✓ revenue_stability
  ✓ engagement_level

Feature Statistics:
       lifetime_spend  avg_monthly_spend  retention_rate  revenue_stability  engagement_level
count     5636.000000        5636.000000     5636.000000        5636.000000       5636.000000
mean      1555.610690          61.967912        0.687515           0.191527          0.380292
std       1605.453955          28.971378        0.251529           0.298371          0.227863
min         18.800000          18.250000        0.077906           0.015514          0.008333
25%        265.337500          31.175000        0.490532           0.025272          0.179762
50%        940.075000          69.000000        0.743547           0.050433          0.351190
75%       2448.087500          85.700000        0.918269           0.172137          0.546429
max       704

In [7]:
#PHASE 5 PART 2 : CLV MODELING - BUILD REGRESSION MODEL

print("PHASE 5 PART 2: CLV SCORING MODEL ")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

target = 'lifetime_spend'
clv_model_features = ['tenure','num_services','avg_monthly_spend','engagement_level','retention_rate','revenue_stability','contract_ordinal']


PHASE 5 PART 2: CLV SCORING MODEL 


In [8]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ==========================================
# 1. Define Features & Target
# ==========================================
target = 'lifetime_spend'
clv_model_features = ['tenure', 'num_services', 'avg_monthly_spend', 
                      'engagement_level', 'retention_rate', 'revenue_stability', 'contract_ordinal']

X = clv_data[clv_model_features]
y = clv_data[target]

# Train/Test Split (to evaluate model performance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features for Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.transform(X) # Need this to generate scores for everyone

# ==========================================
# 2 & 3. Build & Train Regression Model
# ==========================================
# Using Linear Regression for maximum interpretability
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Quick evaluation
y_pred_test = lr_model.predict(X_test_scaled)
print(f"Model R-squared on Test Set: {r2_score(y_test, y_pred_test):.4f}")

# ==========================================
# 4. Generate Predicted CLV Scores
# ==========================================
# Predict on the entire dataset to get a score for every customer
raw_predictions = lr_model.predict(X_all_scaled)

# Since linear regression can sometimes predict negative values, we clip them at 0
clv_data['raw_clv'] = np.clip(raw_predictions, a_min=0, a_max=None)

# ==========================================
# 5 & 6. Normalize CLV Score & Add Column
# ==========================================
# Scale the raw predicted values between 0 and 1 for easier interpretation/ranking
min_max_scaler = MinMaxScaler()
clv_data['clv_score'] = min_max_scaler.fit_transform(clv_data[['raw_clv']])

print("\n--- Summary of New CLV Score ---")
display(clv_data['clv_score'].describe())

# ==========================================
# 7. Reasoning / Assumptions
# ==========================================
print("""
--- CLV Modeling Assumptions & Reasoning ---
Why this is a proxy (not perfect CLV):
- True CLV rigorously calculates the net present value of all future cash flows (incorporating discount rates, exact margins, etc.) over a customer's entire lifetime. 
- We are using a proxy approach by predicting overall 'lifetime_spend' based on current engagement, tenure, and retention probability. This gives us a highly actionable index/score that correlates with true future value, but it is not an exact dollar amount of future net profit.

Assumptions made:
- We assume that historical relationships between tenure, spend, and engagement apply to future behavior.
- By clipping predictions at 0, we assume no customer has a strictly negative lifetime value.
- Simplicity over perfection: A simple Linear Regression model avoids overfitting and gives us a usable, interpretable score that marketing teams can easily understand.
""")


Model R-squared on Test Set: 0.9093

--- Summary of New CLV Score ---


count    5636.000000
mean        0.289526
std         0.256401
min         0.000000
25%         0.061427
50%         0.226491
75%         0.477170
max         1.000000
Name: clv_score, dtype: float64


--- CLV Modeling Assumptions & Reasoning ---
Why this is a proxy (not perfect CLV):
- True CLV rigorously calculates the net present value of all future cash flows (incorporating discount rates, exact margins, etc.) over a customer's entire lifetime. 
- We are using a proxy approach by predicting overall 'lifetime_spend' based on current engagement, tenure, and retention probability. This gives us a highly actionable index/score that correlates with true future value, but it is not an exact dollar amount of future net profit.

Assumptions made:
- We assume that historical relationships between tenure, spend, and engagement apply to future behavior.
- By clipping predictions at 0, we assume no customer has a strictly negative lifetime value.
- Simplicity over perfection: A simple Linear Regression model avoids overfitting and gives us a usable, interpretable score that marketing teams can easily understand.



In [9]:
# ══════════════════════════════════════════════════════════════════════════
# PHASE 5 PART 3: CLV SEGMENTATION
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd

# 1. & 4. Segment Customers / Create clv_segment column
# Using pd.qcut to divide the data into 3 roughly equal buckets (tertiles)
clv_data['clv_segment'] = pd.qcut(
    clv_data['clv_score'], 
    q=[0, 0.333, 0.667, 1.0], 
    labels=['Low CLV', 'Medium CLV', 'High CLV']
)

# 2. Analyze each segment (Calculate Averages)
key_metrics = ['avg_monthly_spend', 'lifetime_spend', 'tenure', 'churn_probability', 'engagement_level']
segment_summary = clv_data.groupby('clv_segment')[key_metrics].mean().round(3)

print("--- CLV Segment Summary Statistics ---")
display(segment_summary)

print("\n--- Segment Size Distribution ---")
print(clv_data['clv_segment'].value_counts(sort=False).to_frame(name='Customer Count'))

# 3. & 5. Reasoning and Validation
print("""
--- Why Segmentation is Needed (vs. Raw Scores) ---
1. Actionability: Translating a raw score (e.g., 0.63) into a category ("High CLV") makes it 
   actionable for marketing and retention teams. 
   - High CLV: White-glove treatment, loyalty programs, VIP retention.
   - Medium CLV: Nurturing campaigns, cross-selling to grow their value.
   - Low CLV: Cost-to-serve optimization, automated interventions, low-cost marketing.
   
2. Communication: Business stakeholders understand discrete tiers much better than continuous 
   probability or normalized scales.

--- Validation ---
As seen in the summary statistics above, the segments behave intuitively:
✓ High CLV customers predictably have the longest average tenure.
✓ High CLV customers have the largest lifetime spend and monthly spend.
✓ Low CLV customers have the highest average churn probability.
✓ Engagement level scales upward as CLV increases.
""")

--- CLV Segment Summary Statistics ---


,avg_monthly_spend,lifetime_spend,tenure,churn_probability,engagement_level
clv_segment,,,,,
Low CLV,40.839,212.960,6.674,0.363,0.151
Medium CLV,63.487,978.835,20.290,0.352,0.348
High CLV,81.573,3476.573,43.392,0.222,0.642



--- Segment Size Distribution ---
             Customer Count
clv_segment                
Low CLV                1877
Medium CLV             1882
High CLV               1877

--- Why Segmentation is Needed (vs. Raw Scores) ---
1. Actionability: Translating a raw score (e.g., 0.63) into a category ("High CLV") makes it 
   actionable for marketing and retention teams. 
   - High CLV: White-glove treatment, loyalty programs, VIP retention.
   - Medium CLV: Nurturing campaigns, cross-selling to grow their value.
   - Low CLV: Cost-to-serve optimization, automated interventions, low-cost marketing.

2. Communication: Business stakeholders understand discrete tiers much better than continuous 
   probability or normalized scales.

--- Validation ---
As seen in the summary statistics above, the segments behave intuitively:
✓ High CLV customers predictably have the longest average tenure.
✓ High CLV customers have the largest lifetime spend and monthly spend.
✓ Low CLV customers have the hig

In [11]:
# ══════════════════════════════════════════════════════════════════════════
# PHASE 6 PART 2: SEGMENTATION ENGINE IMPLEMENTATION
# ══════════════════════════════════════════════════════════════════════════

import pandas as pd

# 1. Define explicit thresholds based on our Phase 6 Part 1 business logic
# Why 0.50 for churn? Anyone with >50% probability is mathematically more likely to leave than stay.
CHURN_HIGH_THRESH = 0.50

# Why 0.30 for low churn? Represents safe, stable customers with <30% risk.
CHURN_LOW_THRESH = 0.30

# Why 0.50 for engagement? It's our normalized middle-ground for tenure + service adoption.
ENGAGEMENT_HIGH_THRESH = 0.50


# 2 & 3. Define the Segmentation engine function
def assign_segment_and_action(row):
    """
    Applies transparent business rules to classify each customer and assign an action.
    """
    
    # ── RULE A: RETENTION TARGETS ──
    # Criteria: High flight risk AND High historical/future value.
    # Logic: These are premium customers on the verge of leaving. Losing them hurts MRR significantly.
    if row['churn_probability'] > CHURN_HIGH_THRESH and row['clv_segment'] == 'High CLV':
        return pd.Series(['Retention Target', 'Offer Discount'])
    
    # ── RULE C: NON-TARGETS ──
    # Criteria: High flight risk BUT Low value.
    # Logic: It costs more in incentives to save them than they will generate in future profits. Let them churn.
    elif row['churn_probability'] > CHURN_HIGH_THRESH and row['clv_segment'] == 'Low CLV':
        return pd.Series(['Non-Target', 'No Action'])
    
    # ── RULE B: LOYALTY / UPSELL TARGETS ──
    # Criteria: Safe (low churn risk) AND highly sticky/engaged.
    # Logic: These customers love the service. Offering discounts here wastes money. Instead, cross-sell.
    elif row['churn_probability'] < CHURN_LOW_THRESH and row['engagement_level'] > ENGAGEMENT_HIGH_THRESH:
        return pd.Series(['Loyalty/Upsell Target', 'Upsell / Reward'])
    
    # ── DEFAULT / MIDDLE TIER ──
    # Criteria: Everyone else (Medium CLV, moderate churn risk).
    # Logic: Standard marketing drip campaigns. No expensive interventions required yet.
    else:
        return pd.Series(['General Monitoring', 'Standard Nurture'])


# 4 & 6. Apply logic to create new columns 'customer_segment' and 'recommended_action'
print("Running segmentation engine...\n")
clv_data[['customer_segment', 'recommended_action']] = clv_data.apply(assign_segment_and_action, axis=1)

# Display the results
print("="*60)
print("CUSTOMER SEGMENTS PRODUCED:")
print("="*60)
segment_counts = clv_data['customer_segment'].value_counts()
for seg, count in segment_counts.items():
    print(f"{seg:25s} : {count:,} customers ({count/len(clv_data)*100:.1f}%)")

print("\n" + "="*60)
print("RECOMMENDED ACTIONS PRODUCED:")
print("="*60)
action_counts = clv_data['recommended_action'].value_counts()
for action, count in action_counts.items():
    print(f"{action:25s} : {count:,} customers ({count/len(clv_data)*100:.1f}%)")

print("\n✓ Segmentation Engine successfully applied.")

Running segmentation engine...

CUSTOMER SEGMENTS PRODUCED:
General Monitoring        : 3,629 customers (64.4%)
Loyalty/Upsell Target     : 1,240 customers (22.0%)
Non-Target                : 613 customers (10.9%)
Retention Target          : 154 customers (2.7%)

RECOMMENDED ACTIONS PRODUCED:
Standard Nurture          : 3,629 customers (64.4%)
Upsell / Reward           : 1,240 customers (22.0%)
No Action                 : 613 customers (10.9%)
Offer Discount            : 154 customers (2.7%)

✓ Segmentation Engine successfully applied.


In [12]:
# ══════════════════════════════════════════════════════════════════════════
# PHASE 6 PART 3: PRIORITIZATION & FINAL OUTPUT
# ══════════════════════════════════════════════════════════════════════════

# 1. Define Priority Score
# priority_score = probability of leaving * relative value of the customer
# This naturally bubbles up high-value flight-risk customers to the top.
clv_data['priority_score'] = clv_data['churn_probability'] * clv_data['clv_score']

# 2 & 3 & 4. Rank and sort the dataset to create the final business-ready table
columns_for_business = [
    'customerID', 
    'churn_probability', 
    'clv_score', 
    'customer_segment', 
    'recommended_action', 
    'priority_score'
]

# Sort descending so highest priority (biggest expected loss) is at index 0
final_business_table = clv_data[columns_for_business].sort_values(
    by='priority_score', 
    ascending=False
).reset_index(drop=True)

# 5. Display the top 20 high-priority customers
print("="*80)
print("TOP 20 HIGH-PRIORITY CUSTOMERS FOR IMMEDIATE ACTION")
print("="*80)
display(final_business_table.head(20))

# 6. Business application and cost reasoning
print("""
--- HOW BUSINESS TEAMS SHOULD USE THIS TABLE ---
1. Retention Team (Call Center): Start at row 0 and work downwards. These are the 
   customers who are extremely likely to leave AND are highly valuable. A saved account
   here protects significant MRR (Monthly Recurring Revenue).
   
2. Marketing Team (Automated): Filter for the 'Loyalty/Upsell Target' segment to build 
   targeted cross-sell email campaigns, avoiding sending discounts to happy people.

--- HOW THIS REDUCES MARKETING COSTS ---
Without this prioritization, a retention budget acts like a scattergun—spending $20 
on a promotion for a customer who only generates $10 in lifetime profit, or offering 
discounts to customers who weren't going to leave anyway. 

By using the Priority Score (Risk × Value), the business surgically applies its retention 
budget ONLY where the ROI is positive, vastly reducing wasted promotional spend.
""")

# Optional: Save to CSV for the business units to ingest into their CRM/marketing tools
final_business_table.to_csv('final_customer_action_list.csv', index=False)
print("\n✓ Saved final action list to 'final_customer_action_list.csv'")

TOP 20 HIGH-PRIORITY CUSTOMERS FOR IMMEDIATE ACTION


,customerID,churn_probability,clv_score,customer_segment,recommended_action,priority_score
0,5647-FXOTP,0.570503,0.850401,Retention Target,Offer Discount,0.485157
1,6377-WHAOX,0.518080,0.867367,Retention Target,Offer Discount,0.449365
2,0946-CLJTI,0.570943,0.785337,Retention Target,Offer Discount,0.448382
3,7752-XUSCI,0.512908,0.867463,Retention Target,Offer Discount,0.444928
4,7901-TBKJX,0.565514,0.779065,Retention Target,Offer Discount,0.440572
5,8785-CJSHH,0.523515,0.827175,Retention Target,Offer Discount,0.433039
6,2675-DHUTR,0.545192,0.794128,Retention Target,Offer Discount,0.432953
7,3791-LGQCY,0.537942,0.784888,Retention Target,Offer Discount,0.422225
8,2277-DJJDL,0.507797,0.827209,Retention Target,Offer Discount,0.420054
9,6646-VRFOL,0.539935,0.772689,Retention Target,Offer Discount,0.417202



--- HOW BUSINESS TEAMS SHOULD USE THIS TABLE ---
1. Retention Team (Call Center): Start at row 0 and work downwards. These are the 
   customers who are extremely likely to leave AND are highly valuable. A saved account
   here protects significant MRR (Monthly Recurring Revenue).

2. Marketing Team (Automated): Filter for the 'Loyalty/Upsell Target' segment to build 
   targeted cross-sell email campaigns, avoiding sending discounts to happy people.

--- HOW THIS REDUCES MARKETING COSTS ---
Without this prioritization, a retention budget acts like a scattergun—spending $20 
on a promotion for a customer who only generates $10 in lifetime profit, or offering 
discounts to customers who weren't going to leave anyway. 

By using the Priority Score (Risk × Value), the business surgically applies its retention 
budget ONLY where the ROI is positive, vastly reducing wasted promotional spend.


✓ Saved final action list to 'final_customer_action_list.csv'
